# 🛒 电商营销大模型训练 Pipeline v2.0

基于 **Qwen2.5** 系列模型，使用电商营销与销售数据集进行完整训练。

## 模块链条总览

```
┌─────────────┐   ┌──────────────┐   ┌──────────────────┐   ┌─────────────┐
│  Module 0    │──▶│  Module 1     │──▶│  Module 2         │──▶│  Module 3    │
│  环境配置    │   │  数据生成     │   │  数据质量清洗     │   │  Continue PT │
│  依赖安装    │   │  + 多轮对话   │   │  去重/过滤/排序   │   │  增量预训练  │
└─────────────┘   └──────────────┘   └──────────────────┘   └──────┬──────┘
                                                                    │
                                                                    ▼
┌─────────────┐   ┌──────────────┐   ┌──────────────────┐   ┌─────────────┐
│  Module 7    │◀──│  Module 6     │◀──│  Module 5         │◀──│  Module 4    │
│  最终测试    │   │  综合评估     │   │  DPO偏好对齐      │   │  SFT微调    │
│  效果对比    │   │  多维度评分   │   │  Adaptive-β/ORPO  │   │  LLaMA-Factory│
└─────────────┘   └──────────────┘   └──────────────────┘   └─────────────┘
```

### v2.0 主要更新

| 模块 | 更新内容 |
|------|----------|
| SFT 微调 | 从自定义脚本切换为 **LLaMA-Factory** 框架 |
| 多轮对话 | 新增 **Self-Play + Evol-Instruct** 多轮对话数据合成 |
| 对齐算法 | 移除 SimPO，保留 **Adaptive-β DPO** + **ORPO** |
| Pipeline | 重新整理模块顺序，形成完整链条 |

> 💡 **推荐配置**：单卡 A100 40GB 或 4090 24GB，使用 Qwen2.5-7B-Instruct

---

## Module 0️⃣ 环境配置

In [ ]:
# ============================================================
# 0.1 安装核心依赖
# ============================================================
!pip install transformers>=4.46.0 \
             peft>=0.13.0 \
             trl>=0.12.0 \
             datasets>=3.1.0 \
             accelerate>=1.1.0 \
             bitsandbytes>=0.44.1 \
             sentencepiece \
             tensorboard \
             tqdm \
             ollama \
             pyyaml \
             -q

# 安装 LLaMA-Factory（SFT 训练框架）
!pip install llamafactory -q

In [2]:
# ============================================================
# 0.2 验证 GPU 环境
# ============================================================
import torch
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 型号: {torch.cuda.get_device_name(0)}")
    print(f"显存大小: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")


libgomp: Invalid value for environment variable OMP_NUM_THREADS


PyTorch 版本: 2.3.0+cu121
CUDA 可用: True
GPU 型号: NVIDIA A100-PCIE-40GB


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [3]:
# ============================================================
# 0.3 全局路径与环境变量配置
# ============================================================
import os

# HuggingFace 缓存设置（根据你的磁盘空间调整）
os.environ["HF_HOME"] = "/root/autodl-tmp/hf_cache"
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"   # 国内镜像加速
os.environ["MODELSCOPE_CACHE"] = "/root/autodl-tmp/modelscope_cache"

# 全局模型配置
BASE_MODEL       = "Qwen/Qwen2.5-7B-Instruct"   # 基座模型
PT_BASE_MODEL    = "Qwen/Qwen2.5-7B"             # PT用Base版
WORK_DIR         = "/root/autodl-tmp"             # 数据盘工作目录

print(f"✅ 环境配置完成")
print(f"   基座模型: {BASE_MODEL}")
print(f"   工作目录: {WORK_DIR}")

✅ 环境配置完成
   基座模型: Qwen/Qwen2.5-7B-Instruct
   工作目录: /root/autodl-tmp


---

## Module 1️⃣ 数据生成

### 1.1 单轮 SFT + DPO 数据生成

使用本地 Ollama + Qwen2.5 作为数据生成器，结合论文技术栈：
- GLAN（分类树自动构建）
- Self-Instruct（指令自举）
- Evol-Instruct（复杂度进化）
- InsTag（多样性过滤）

**前置条件**：
```bash
ollama pull qwen2.5:7b
```

In [5]:
from openai import OpenAI
client = OpenAI(api_key="sk-655ad4c737eb40239b5fa7813aef62a7", base_url="https://api.deepseek.com")

# 列出可用模型
models = client.models.list()
for m in models.data:
    print(m.id)

deepseek-chat
deepseek-reasoner


In [1]:
# ============================================================
# 1.1 生成单轮 SFT + DPO 数据
# ============================================================

# 第一次：构建分类树 + 生成数据
# !python generate_ecommerce_dataset_v3.py \
#     --model qwen2.5:7b --sft 500 --dpo 200

# 后续复用已有分类树（跳过 GLAN 构建步骤，更快）
!python generate_ecommerce_dataset_v3.py \
    --model deepseek-chat \
    --load_taxonomy data/taxonomy.json \
    --sft 500 --dpo 200


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🛒 电商数据集生成器 v3.0 — LLM知识驱动
  模型: deepseek-chat
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  论文技术栈:
    GLAN       分类树自动构建（ICLR 2024）
    Self-Instruct 指令自举生成（ACL 2023）
    Evol-Instruct 复杂度进化（ICLR 2024）
    InsTag     意图标签多样性过滤（ICLR 2024）
    D3/NovelSum  novelty去重（IJCAI 2025 / 2024）
✅ 模型后端: deepseek-chat (DeepSeek API)
  ✅ 模型连通性测试通过

[GLAN] 加载已有分类树: data/taxonomy.json
  加载 16 个叶节点

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Stage 1 ▶ PT 预训练文本  |  数量=20
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PT文章: 100%|█████████████████████████████████| 16/16 [17:58<00:00, 67.39s/it]
  ✅ 保存 16 篇 → data/pretrain/ecommerce_pretrain.txt

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Stage 2 ▶ SFT 指令微调  |  目标=500
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📌 从断点恢复: 已有 100 条，继续生成...

  [Phase A] GLAN+Self-Instruct 生成初始指令 (目标约 350 条)
  Phase A: 100%|████████████████████

IOStream.flush timed out


  Phase B 生成回复: 100%|███████████████████| 300/300 [4:13:47<00:00, 50.76s/it]
  Phase B 完成: 300 条

  合并后总量: 650 条

  [Phase C] InsTag 多样性选择...
  [InsTag] 为样本生成意图标签...
  打标签: 100%|███████████████████████████████| 650/650 [02:06<00:00,  5.12it/s]
  [InsTag] 选择 308 条，覆盖标签类型: 185

  [Phase D] Novelty 去重过滤...

  ✅ SFT 最终保存: 296 条 → data/finetune/ecommerce_sft.jsonl

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Stage 3 ▶ DPO 偏好对比  |  数量=200
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  从GLAN分类树生成额外 80 个问题...
  生成DPO问题: 100%|████████████████████████████| 16/16 [02:27<00:00,  9.21s/it]
  DPO问题总数: 125
  生成DPO对: 100%|██████████████████████████| 125/125 [1:30:05<00:00, 43.25s/it]
  ✅ 保存 125 对（跳过 0）→ data/reward/ecommerce_dpo.jsonl

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📊 数据集质量报告  （v3.0 LLM知识驱动）
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  🗂  GLAN 分类树
     功能模块数:   8
     子模块数:     16
     叶节点总数:   16
     模块列表: 营销文案创作, 商品运营管理, 用户运营与私域, 促销活

### 1.2 多轮对话数据生成（新增）

使用 **Self-Play + Evol-Instruct** 方案自动合成电商多轮对话：
- 覆盖 6 大场景：售前咨询、售后处理、运营策划、数据分析讨论、直播策划、砍价博弈
- 对话进化策略：深度追问、场景切换、反驳纠错、信息补充
- 多维度质量过滤：轮次、长度、重复度、实质性

In [2]:
# ============================================================
# 1.2 生成多轮对话数据
# ============================================================
!python multiturn_dialogue.py \
    --model deepseek-chat \
    --num_dialogues 200 \
    --max_turns 6 \
    --min_turns 3 \
    --output_dir ./data/multiturn \
    --llamafactory_dir ./data/llamafactory_sft

2026-03-17 00:32:05,299 - INFO - 📌 从断点恢复：已有 200 条
生成多轮对话: 100%|███████████████████████████████████| 200/200 [00:00<?, ?it/s]
2026-03-17 00:32:05,561 - INFO - ✅ 多轮对话数据集生成完成：200 条 → ./data/multiturn/ecommerce_multiturn.jsonl
2026-03-17 00:32:05,562 - INFO - 
📊 数据集统计:
2026-03-17 00:32:05,562 - INFO -   总对话数: 200
2026-03-17 00:32:05,562 - INFO -   平均轮次: 4.6
2026-03-17 00:32:05,562 - INFO -   运营策划: 34 条
2026-03-17 00:32:05,562 - INFO -   售后处理: 34 条
2026-03-17 00:32:05,562 - INFO -   砍价博弈: 33 条
2026-03-17 00:32:05,562 - INFO -   直播策划: 33 条
2026-03-17 00:32:05,562 - INFO -   数据分析讨论: 33 条
2026-03-17 00:32:05,562 - INFO -   售前咨询: 33 条
2026-03-17 00:32:05,592 - INFO - ✅ LLaMA-Factory 格式转换完成：200 条 → ./data/llamafactory_sft/ecommerce_multiturn.json


In [3]:
# ============================================================
# 1.3 数据预览
# ============================================================
import os, json
from pathlib import Path

print("=" * 60)
print("📁 生成文件列表")
print("=" * 60)
for d in ["data/pretrain", "data/finetune", "data/reward", "data/multiturn"]:
    files = list(Path(d).glob("*")) if Path(d).exists() else []
    for f in files:
        if f.is_file():
            size = f.stat().st_size / 1024
            print(f"  {f}  ({size:.1f} KB)")

# 预览 SFT 数据
print("\n" + "=" * 60)
print("📋 SFT 数据预览（前2条）")
print("=" * 60)
sft_file = "data/finetune/ecommerce_sft.jsonl"
if Path(sft_file).exists():
    with open(sft_file, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 2: break
            d = json.loads(line)
            print(f"\n[样本 {i+1}]")
            print(f"  Instruction: {d['instruction'][:80]}...")
            print(f"  Output:      {d['output'][:120]}...")

# 预览多轮对话
print("\n" + "=" * 60)
print("💬 多轮对话预览（前1条）")
print("=" * 60)
mt_file = "data/multiturn/ecommerce_multiturn.jsonl"
if Path(mt_file).exists():
    with open(mt_file, encoding="utf-8") as f:
        d = json.loads(f.readline())
        convs = d.get("conversations", [])
        meta = d.get("metadata", {})
        print(f"  场景: {meta.get('category', '未知')}")
        print(f"  轮次: {meta.get('num_turns', 0)}")
        for j, c in enumerate(convs[:4]):
            role = "🧑 用户" if c['from'] == 'human' else "🤖 助手"
            print(f"  [{role}]: {c['value'][:100]}...")

# 预览 DPO 数据
print("\n" + "=" * 60)
print("⚖️  DPO 数据预览（前1条）")
print("=" * 60)
dpo_file = "data/reward/ecommerce_dpo.jsonl"
if Path(dpo_file).exists():
    with open(dpo_file, encoding="utf-8") as f:
        d = json.loads(f.readline())
        print(f"  问题:    {d['instruction'][:80]}")
        print(f"  Chosen:  {d['output'][0][:150]}...")
        print(f"  Rejected:{d['output'][1][:100]}...")

📁 生成文件列表
  data/pretrain/ecommerce_pretrain.txt  (104.7 KB)
  data/finetune/ecommerce_sft.jsonl  (1276.0 KB)
  data/reward/ecommerce_dpo.jsonl  (676.8 KB)
  data/multiturn/ecommerce_multiturn_checkpoint.jsonl  (1884.2 KB)
  data/multiturn/ecommerce_multiturn.jsonl  (1884.2 KB)

📋 SFT 数据预览（前2条）

[样本 1]
  Instruction: 作为拼多多家居用品店铺运营专家，当前店铺日均访客2000，主要依赖平台活动流量，付费广告ROI为1.2。请设计一套为期1个月的流量获取与广告进阶策略，目标将日均...
  Output:      # 拼多多家居用品店铺流量与广告进阶策略（1个月）

## 一、流量渠道拓展方案
**1. 店铺直播常态化（低成本流量入口）**
- **结合方式**：家居用品适合场景化演示。每日固定时段（如20:00-22:00）直播，展示商品使用场景（...

[样本 2]
  Instruction: 作为抖音女装直播间的运营负责人，你发现直播间平均在线人数稳定在2000人，但平均观看时长仅2分钟，远低于行业平均5分钟，且商品点击转化率仅8%。现需设计一套直播...
  Output:      针对您面临的挑战，我设计了一套名为 **“沉浸式节奏卡点”** 的直播优化策略。该策略的核心在于：**通过精心设计的内容节奏和低成本的强互动环节，将2分钟的“路过式观看”转化为4分钟的“沉浸式体验”，从而自然提升商品点击意愿。**

以下为...

💬 多轮对话预览（前1条）
  场景: 砍价博弈
  轮次: 6
  [🧑 用户]: 您好，请问这款高端耳机的初次报价是多少呢？...
  [🤖 助手]: 您好！关于这款高端耳机的初次报价，我们给出的市场参考价是2999元。不过，考虑到您对这款产品的高度关注，我们可以为您争取一个更优惠的价格。通常我们会根据客户的购买量和合作情况提供一定的折扣。例如，如果...
  [🧑 用户]: 那如果我想一次性购买五副耳机，能否再争取

---

## Module 2️⃣ 数据质量清洗

对生成的 SFT 数据进行质量管控：
- 去重（SimHash / 编辑距离）
- 质量过滤（字符多样性、最小长度）
- IFD 难度打分
- 课程学习排序（Curriculum Learning）

In [1]:
# ============================================================
# 2.1 SFT 数据清洗
# ============================================================
!python data_quality_pipeline.py \
    --sft_input data/finetune/ecommerce_sft.jsonl \
    --sft_output data/cleaned/sft_cleaned.jsonl

2026-03-17 14:26:57,516 - INFO - [SFT Pipeline] 读取数据: data/finetune/ecommerce_sft.jsonl
2026-03-17 14:26:57,542 - INFO - 原始样本数: 296
2026-03-17 14:26:57,542 - INFO - Step 1: 去重...
2026-03-17 14:27:59,316 - INFO - 去重后: 296 (0 删除)
2026-03-17 14:27:59,317 - INFO - Step 2: 质量过滤...
2026-03-17 14:27:59,367 - INFO - 过滤后: 62 (234 删除)
2026-03-17 14:27:59,367 - INFO - Step 3: IFD难度打分...
2026-03-17 14:27:59,371 - INFO - Step 5: 课程排序 (策略: bucket)...
2026-03-17 14:27:59,374 - INFO - 已保存至: data/cleaned/sft_cleaned.jsonl

📊 数据质量流水线报告

[SFT]
  pipeline: SFT
  input_path: data/finetune/ecommerce_sft.jsonl
  output_path: data/cleaned/sft_cleaned.jsonl
  stats:
    raw: 296
    after_dedup: 296
    after_filter: 62
    final: 62
    retention_rate: 20.9%
  dedup_stats:
    total_processed: 296
    duplicates_removed: 0
    dedup_rate: 0.0%
    remaining: 296
  filter_stats:
    low_char_diversity: 219
    passed: 62
    output_too_long: 1
    output_too_few_words: 14
  difficulty_distribution:
    easy (0

---

## Module 3️⃣ Continue PreTraining (PT)（可选）

在电商领域文本上进行增量预训练，注入电商领域知识。

> 💡 此阶段**可选**，如果数据量较少（<10万字），可直接跳到 Module 4

In [ ]:
# ============================================================
# 3.1 PT 训练配置
# ============================================================
PT_OUTPUT_DIR  = f"{WORK_DIR}/outputs-ecom-pt-v1"
PT_BATCH_SIZE  = 2   # 24GB→2, 40GB→4
PT_MAX_SAMPLES = 10000

print(f"PT 配置：model={PT_BASE_MODEL}")

In [ ]:
# ============================================================
# 3.2 运行 PT 训练
# ============================================================
!python pretraining.py \
    --model_name_or_path {PT_BASE_MODEL} \
    --train_file_dir ./data/pretrain \
    --validation_file_dir ./data/pretrain \
    --per_device_train_batch_size {PT_BATCH_SIZE} \
    --per_device_eval_batch_size {PT_BATCH_SIZE} \
    --do_train \
    --do_eval \
    --use_peft True \
    --seed 42 \
    --bf16 \
    --max_train_samples {PT_MAX_SAMPLES} \
    --max_eval_samples 10 \
    --num_train_epochs 1 \
    --learning_rate 2e-4 \
    --warmup_ratio 0.05 \
    --weight_decay 0.01 \
    --logging_steps 10 \
    --eval_steps 50 \
    --eval_strategy steps \
    --save_steps 50 \
    --save_strategy steps \
    --save_total_limit 3 \
    --gradient_accumulation_steps 4 \
    --block_size 512 \
    --output_dir {PT_OUTPUT_DIR} \
    --overwrite_output_dir \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --device_map auto \
    --report_to tensorboard \
    --gradient_checkpointing True

In [ ]:
# ============================================================
# 3.3 合并 PT LoRA 权重
# ============================================================
!python merge_peft_adapter.py \
    --base_model {PT_BASE_MODEL} \
    --lora_model {PT_OUTPUT_DIR} \
    --output_dir {WORK_DIR}/merged-ecom-pt

print("✅ PT 阶段完成，合并模型已保存到 merged-ecom-pt/")

---

## Module 4️⃣ Supervised Fine-Tuning (SFT) — LLaMA-Factory

使用 **LLaMA-Factory** 框架进行监督微调，替代原有的自定义 SFT 脚本。

### LLaMA-Factory 优势
- 内置 100+ 模型适配，开箱即用
- 原生支持 FlashAttention-2、Unsloth 加速
- 支持 sharegpt 多轮对话训练格式
- YAML 配置，一键训练
- 内置模型导出（替代 merge_peft_adapter.py）

In [4]:
# ============================================================
# 4.1 SFT 训练配置
# ============================================================

# 如果做了 Stage 3（PT），使用 PT 后的模型：
# SFT_BASE_MODEL = f"{WORK_DIR}/merged-ecom-pt"

# 未做 PT，直接用 Instruct 版：
SFT_BASE_MODEL  = BASE_MODEL
SFT_OUTPUT_DIR  = f"{WORK_DIR}/outputs-ecom-sft-v2"
SFT_MERGED_DIR  = f"{WORK_DIR}/merged-ecom-sft"
SFT_EPOCHS      = 3
SFT_BATCH_SIZE  = 4
SFT_LR          = 2e-5

print(f"SFT 配置：")
print(f"  基座模型: {SFT_BASE_MODEL}")
print(f"  训练框架: LLaMA-Factory")
print(f"  Epochs:   {SFT_EPOCHS}")
print(f"  数据:     单轮 SFT + 多轮对话")

SFT 配置：
  基座模型: Qwen/Qwen2.5-7B-Instruct
  训练框架: LLaMA-Factory
  Epochs:   3
  数据:     单轮 SFT + 多轮对话


In [5]:
# ============================================================
# 4.2 数据格式转换 → LLaMA-Factory 格式
# ============================================================
# 转换单轮 SFT 数据 + 多轮对话数据为 LLaMA-Factory sharegpt 格式
# 同时生成 dataset_info.json

from sft_with_llamafactory import (
    convert_to_llamafactory_format,
    convert_multiturn_to_llamafactory,
    generate_dataset_info,
)

DATASET_DIR = "./data/llamafactory_sft"

# 单轮 SFT 数据转换
convert_to_llamafactory_format(
    input_path="./data/finetune/ecommerce_sft.jsonl",
    output_dir=DATASET_DIR,
    dataset_name="ecommerce_sft",
)

# 多轮对话数据（如果已生成）
import os
if os.path.exists("./data/multiturn/ecommerce_multiturn.jsonl"):
    from multiturn_dialogue import convert_to_llamafactory_multiturn
    convert_to_llamafactory_multiturn(
        input_path="./data/multiturn/ecommerce_multiturn.jsonl",
        output_dir=DATASET_DIR,
    )

# 生成 dataset_info.json（LLaMA-Factory 必须）
generate_dataset_info(dataset_dir=DATASET_DIR)

2026-03-17 14:28:35,430 - INFO - ✅ 转换完成：296 条 → ./data/llamafactory_sft/ecommerce_sft.json
2026-03-17 14:28:35,469 - INFO - ✅ LLaMA-Factory 格式转换完成：200 条 → ./data/llamafactory_sft/ecommerce_multiturn.json
2026-03-17 14:28:35,471 - INFO - ✅ dataset_info.json 已生成：./data/llamafactory_sft/dataset_info.json
2026-03-17 14:28:35,472 - INFO -    包含数据集：['ecommerce_multiturn', 'ecommerce_sft', 'dataset_info']


'./data/llamafactory_sft/dataset_info.json'

In [6]:
# ============================================================
# 4.3 生成 LLaMA-Factory 训练 YAML 配置
# ============================================================
from sft_with_llamafactory import generate_training_yaml

# 检测可用数据集
from pathlib import Path
dataset_names = [f.stem for f in Path(DATASET_DIR).glob("*.json") if f.stem != "dataset_info"]
print(f"可用数据集: {dataset_names}")

config_path = generate_training_yaml(
    model_name_or_path=SFT_BASE_MODEL,
    dataset_dir=DATASET_DIR,
    dataset_names=",".join(dataset_names),
    output_dir=SFT_OUTPUT_DIR,
    config_path="./train_sft.yaml",
    # 训练超参
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=SFT_LR,
    max_length=2048,
    # LoRA
    use_lora=True,
    lora_rank=16,
    lora_alpha=32,
    lora_dropout=0.05,
    # 其他
    bf16=True,
    gradient_checkpointing=True,
)

# 查看生成的配置
print("\n" + "=" * 60)
print("📄 训练配置 (train_sft.yaml)")
print("=" * 60)
with open(config_path, "r") as f:
    print(f.read())

2026-03-17 14:28:39,223 - INFO - ✅ 训练配置已生成：./train_sft.yaml


可用数据集: ['ecommerce_multiturn', 'ecommerce_sft']

📄 训练配置 (train_sft.yaml)
bf16: true
cutoff_len: 2048
dataset: ecommerce_multiturn,ecommerce_sft
dataset_dir: ./data/llamafactory_sft
do_train: true
eval_steps: 100
eval_strategy: steps
finetuning_type: lora
flash_attn: auto
gradient_accumulation_steps: 2
gradient_checkpointing: true
learning_rate: 2.0e-05
logging_steps: 10
lora_alpha: 32
lora_dropout: 0.05
lora_rank: 16
lora_target: all
lr_scheduler_type: cosine
max_grad_norm: 1.0
model_name_or_path: Qwen/Qwen2.5-7B-Instruct
num_train_epochs: 3
optim: adamw_torch
output_dir: /root/autodl-tmp/outputs-ecom-sft-v2
overwrite_cache: true
overwrite_output_dir: true
per_device_eval_batch_size: 4
per_device_train_batch_size: 4
preprocessing_num_workers: 8
report_to: tensorboard
save_steps: 200
save_total_limit: 3
stage: sft
template: qwen
trust_remote_code: true
val_size: 0.02
warmup_ratio: 0.05
weight_decay: 0.05



In [12]:
%cd /root/autodl-tmp/

/root/autodl-tmp


/root/miniconda3/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [13]:
# ============================================================
# 4.4 启动 LLaMA-Factory SFT 训练
# ============================================================
!llamafactory-cli train /root/autodl-tmp/train_sft.yaml


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS
[INFO|2026-03-17 14:50:40] llamafactory.hparams.parser:468 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|tokenization_utils_base.py:2060] 2026-03-17 14:50:41,624 >> loading file vocab.json from cache at /root/autodl-tmp/hf_cache/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/vocab.json
[INFO|tokenization_utils_base.py:2060] 2026-03-17 14:50:41,625 >> loading file merges.txt from cache at /root/autodl-tmp/hf_cache/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/merges.txt
[INFO|tokenization_utils_base.py:2060] 2026-03-17 14:50:41,625 >> loading file tokenizer.json from cache at /root/autodl-tmp/hf_cache/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/tokenizer.json
[IN

In [ ]:
# ============================================================
# 4.5 导出合并模型（LLaMA-Factory 内置）
# ============================================================
from sft_with_llamafactory import export_model

export_model(
    model_name_or_path=SFT_BASE_MODEL,
    adapter_path=SFT_OUTPUT_DIR,
    output_dir=SFT_MERGED_DIR,
)

print(f"✅ SFT 阶段完成，合并模型已保存到 {SFT_MERGED_DIR}")

In [ ]:
# ============================================================
# 4.6 SFT 效果快速测试
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

sft_model_path = SFT_MERGED_DIR
tokenizer = AutoTokenizer.from_pretrained(sft_model_path, trust_remote_code=True)
model     = AutoModelForCausalLM.from_pretrained(
    sft_model_path, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
model.eval()

def sft_test(question: str, history: list = None) -> str:
    """支持多轮对话测试"""
    messages = [{"role": "system", "content": "你是专业的电商运营和销售顾问。"}]
    if history:
        for h_q, h_a in history:
            messages.append({"role": "user", "content": h_q})
            messages.append({"role": "assistant", "content": h_a})
    messages.append({"role": "user", "content": question})
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
    new_ids = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)

# 单轮测试
test_questions = [
    "帮我写一个「无线蓝牙耳机」的淘宝爆款标题",
    "买家说太贵了要砍价，客服应该怎么回复？",
    "新店铺零销量如何快速打造第一个爆款？",
]
for q in test_questions:
    print(f"\n{'='*50}")
    print(f"❓ 问题：{q}")
    print(f"{'='*50}")
    print(f"💬 回答：{sft_test(q)}")

# 多轮对话测试
print(f"\n{'='*60}")
print("💬 多轮对话测试")
print(f"{'='*60}")

history = []
multi_turn_questions = [
    "我想在抖音开一个卖女装的店铺，应该怎么起步？",
    "预算大概5万块，你觉得够不够？应该怎么分配？",
    "供应链这块我完全没经验，有什么建议？",
]

for q in multi_turn_questions:
    print(f"\n🧑 用户：{q}")
    ans = sft_test(q, history=history)
    print(f"🤖 助手：{ans[:300]}...")
    history.append((q, ans))

In [ ]:
# 释放 SFT 模型显存
del model, tokenizer
torch.cuda.empty_cache()

---

## Module 5️⃣ Direct Preference Optimization (DPO)

使用偏好对比数据进一步对齐模型。提供两种方案：

| 方法 | 特点 | 适用场景 |
|------|------|----------|
| **Adaptive-β DPO** | β自适应调度，训练更稳定 | 通用推荐 |
| **ORPO** | SFT+对齐一步完成，效率最高 | 数据量充足时 |

### 5.1 DPO 数据格式化

In [6]:
# ============================================================
# 5.1 DPO 训练配置
# ============================================================
DPO_BASE_MODEL  = SFT_MERGED_DIR                     # SFT 后的模型
DPO_OUTPUT_DIR  = f"{WORK_DIR}/outputs-ecom-dpo-v2"
DPO_BATCH_SIZE  = 2
DPO_MAX_STEPS   = 180

print(f"DPO 配置：model={DPO_BASE_MODEL}, max_steps={DPO_MAX_STEPS}")

DPO 配置：model=/root/autodl-tmp/merged-ecom-sft, max_steps=180


In [7]:
# ============================================================
# 5.2 DPO 数据格式转换
# ============================================================
import json, os

os.makedirs("./data/reward_formatted", exist_ok=True)

dpo_src = "./data/reward/ecommerce_dpo.jsonl"
dpo_dst = "./data/reward_formatted/ecommerce_dpo_formatted.jsonl"

count = 0
with open(dpo_src, "r", encoding="utf-8") as fin, \
     open(dpo_dst, "w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip(): continue
        item = json.loads(line)
        outputs = item.get("output", [])
        if len(outputs) < 2: continue
        formatted = {
            "system": "你是资深电商运营专家和销售顾问，拥有丰富的实战经验。",
            "question": item.get("instruction", ""),
            "response_chosen": outputs[0],
            "response_rejected": outputs[1],
            "history": []
        }
        fout.write(json.dumps(formatted, ensure_ascii=False) + "\n")
        count += 1

print(f"✅ DPO 数据格式化完成：{count} 条 → {dpo_dst}")

✅ DPO 数据格式化完成：134 条 → ./data/reward_formatted/ecommerce_dpo_formatted.jsonl


### 5.3 方案 A：标准 DPO 训练（推荐）

In [25]:
# ============================================================
# 5.3 标准 DPO 训练
# ============================================================
!python dpo_training.py \
    --model_name_or_path {DPO_BASE_MODEL} \
    --train_file_dir ./data/reward_formatted \
    --validation_file_dir ./data/reward_formatted \
    --per_device_train_batch_size {DPO_BATCH_SIZE} \
    --per_device_eval_batch_size {DPO_BATCH_SIZE} \
    --do_train \
    --do_eval \
    --use_peft True \
    --qlora True \
    --bf16 \
    --max_train_samples 5000 \
    --max_eval_samples 50 \
    --max_steps {DPO_MAX_STEPS} \
    --learning_rate 5e-6 \
    --lr_scheduler_type cosine \
    --warmup_steps 100 \
    --weight_decay 0.05 \
    --logging_steps 1 \
    --eval_steps 20 \
    --eval_strategy steps \
    --save_steps 50 \
    --gradient_accumulation_steps 8 \
    --output_dir {DPO_OUTPUT_DIR} \
    --overwrite_output_dir \
    --target_modules q_proj,k_proj,v_proj,o_proj \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --device_map auto \
    --report_to tensorboard \
    --gradient_checkpointing True \
    --max_source_length 1024 \
    --max_target_length 2048 \
    --template_name qwen


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS
2026-03-12 17:20:41.158 | INFO     | __main__:main:198 - Parse args: ScriptArguments(model_name_or_path='/root/autodl-tmp/merged-ecom-sft', tokenizer_name_or_path=None, load_in_8bit=False, load_in_4bit=False, cache_dir=None, use_fast_tokenizer=False, torch_dtype='bfloat16', device_map='auto', trust_remote_code=True, dataset_name=None, dataset_config_name=None, train_file_dir='./data/reward_formatted', validation_file_dir='./data/reward_formatted', template_name='qwen', per_device_train_batch_size=2, per_device_eval_batch_size=2, max_source_length=1024, max_target_length=2048, min_target_length=4, max_train_samples=5000, max_eval_samples=50, overwrite_cache=False, validation_split_percentage=1, preprocessing_num_workers=4, use_peft=True, qlora=True, target_modules='q_proj,k_proj,v_proj,o_proj', lora_rank=8, lora_dropout=0.05, lora_alpha=16.0, peft_path=None, 

### 5.4 方案 B：Adaptive-β DPO（高级）

In [50]:
# ============================================================
# 5.4 Adaptive-β DPO 训练
# ============================================================
!python advanced_alignment.py \
    --model_name_or_path /root/autodl-tmp/merged-ecom-sft \
    --train_file_dir ./data/reward_formatted \
    --validation_file_dir ./data/reward_formatted \
    --do_train --do_eval \
    --use_peft True --qlora True --bf16 \
    --max_source_length 1024 --max_target_length 2048 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --learning_rate 5e-6 --warmup_steps 10 --max_steps 150 \
    --dpo_beta 0.1 \
    --beta_dpo_mode beta_DPO \
    --beta_dpo_a 0.5 \
    --beta_dpo_mode_weight 0.2 \
    --beta_dpo_ema_gamma 0.95 \
    --lr_scheduler_type constant_with_warmup \
    --output_dir /root/autodl-tmp/outputs-beta-dpo \
    --template_name qwen

2026-03-14 23:21:16.969 | INFO     | __main__:main:275 - Parse args: ScriptArguments(model_name_or_path='/root/autodl-tmp/merged-ecom-sft', tokenizer_name_or_path=None, load_in_8bit=False, load_in_4bit=False, cache_dir=None, use_fast_tokenizer=False, torch_dtype='bfloat16', device_map='auto', trust_remote_code=True, dataset_name=None, dataset_config_name=None, train_file_dir='./data/reward_formatted', validation_file_dir='./data/reward_formatted', template_name='qwen', per_device_train_batch_size=1, per_device_eval_batch_size=1, max_source_length=1024, max_target_length=2048, min_target_length=4, max_train_samples=None, max_eval_samples=None, overwrite_cache=False, validation_split_percentage=1, preprocessing_num_workers=4, use_peft=True, qlora=True, target_modules='q_proj,k_proj,v_proj,o_proj', lora_rank=8, lora_dropout=0.05, lora_alpha=16.0, peft_path=None, do_train=True, do_eval=True, learning_rate=5e-06, lr_scheduler_type='constant_with_warmup', warmup_steps=10, weight_decay=0.05, 

### 5.5 方案 C：ORPO（可选）

In [45]:
# ============================================================
# 5.5 ORPO 训练（SFT+DPO 一体化）直接输出预训练大模型
# ============================================================
!CUDA_VISIBLE_DEVICES=0,1 python orpo_training.py \
    --model_name_or_path Qwen/Qwen2.5-0.5B-Instruct \
    --template_name qwen \
    --train_file_dir ./data/reward_formatted \
    --per_device_train_batch_size 1 \
    --per_device_eval_batch_size 1 \
    --gradient_accumulation_steps 16 \
    --use_peft True \
    --qlora True \
    --do_train True \
    --do_eval True \
    --validation_file_dir ./data/reward_formatted \
    --load_in_4bit True \
    --max_train_samples 1000 \
    --max_eval_samples 50 \
    --max_steps 200 \
    --eval_steps 20 \
    --save_steps 50 \
    --max_source_length 1024 \
    --max_target_length 2048 \
    --output_dir /root/autodl-tmp/outputs-orpo-qwen-v1 \
    --target_modules q_proj,v_proj,k_proj,o_proj \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --bf16 True \
    --report_to tensorboard \
    --remove_unused_columns False \
    --gradient_checkpointing True \
    --orpo_beta 0.1 \
    --cache_dir ./cache

2026-03-14 22:15:13.175 | INFO     | __main__:main:209 - Parse args: ScriptArguments(model_name_or_path='/root/autodl-tmp/merged-ecom-sft', tokenizer_name_or_path=None, load_in_8bit=False, load_in_4bit=True, cache_dir='./cache', use_fast_tokenizer=False, torch_dtype='bfloat16', device_map='auto', trust_remote_code=True, dataset_name=None, dataset_config_name=None, train_file_dir='./data/reward_formatted', validation_file_dir='./data/reward_formatted', template_name='qwen', per_device_train_batch_size=1, per_device_eval_batch_size=1, max_source_length=1024, max_target_length=2048, min_target_length=4, max_train_samples=1000, max_eval_samples=50, overwrite_cache=False, validation_split_percentage=1, preprocessing_num_workers=4, use_peft=True, qlora=True, target_modules='q_proj,v_proj,k_proj,o_proj', lora_rank=8, lora_dropout=0.05, lora_alpha=16.0, peft_path=None, do_train=True, do_eval=True, beta=0.1, learning_rate=0.0005, lr_scheduler_type='cosine', warmup_steps=100, weight_decay=0.05, 

In [71]:
# ============================================================
# 5.6 合并 DPO LoRA 权重
# ============================================================
DPO_MERGED_DIR = f"{WORK_DIR}/merged-ecom-dpo"

!python merge_peft_adapter.py \
    --base_model {DPO_BASE_MODEL} \
    --lora_model {DPO_OUTPUT_DIR} \
    --output_dir {DPO_MERGED_DIR}

print(f"✅ DPO 阶段完成，最终模型已保存到 {DPO_MERGED_DIR}")


libgomp: Invalid value for environment variable OMP_NUM_THREADS
Namespace(base_model='/root/autodl-tmp/merged-ecom-sft', tokenizer_path=None, lora_model='/root/autodl-tmp/outputs-ecom-dpo-v2', resize_emb=False, output_dir='/root/autodl-tmp/merged-ecom-dpo', hf_hub_model_id='', hf_hub_token=None)
Base model: /root/autodl-tmp/merged-ecom-sft
LoRA model: /root/autodl-tmp/outputs-ecom-dpo-v2
Loading LoRA for causal language model
Loading checkpoint shards: 100%|██████████████████| 4/4 [00:05<00:00,  1.34s/it]
Merging with merge_and_unload...
Saving to Hugging Face format...
Done! model saved to /root/autodl-tmp/merged-ecom-dpo
✅ DPO 阶段完成，最终模型已保存到 /root/autodl-tmp/merged-ecom-dpo


In [51]:
# ============================================================
# 5.6 合并 Adaptive-β DPO LoRA 权重
# ============================================================

!python merge_peft_adapter.py \
    --base_model {DPO_BASE_MODEL} \
    --lora_model {WORK_DIR}/outputs-beta-dpo \
    --output_dir {WORK_DIR}/merged-ecom-adaptive-dpo

print(f"✅ DPO 阶段完成，最终模型已保存到 {WORK_DIR}/merged-ecom-adative-dpo")

Namespace(base_model='/root/autodl-tmp/merged-ecom-sft', tokenizer_path=None, lora_model='/root/autodl-tmp/outputs-beta-dpo', resize_emb=False, output_dir='/root/autodl-tmp/merged-ecom-adaptive-dpo', hf_hub_model_id='', hf_hub_token=None)
Base model: /root/autodl-tmp/merged-ecom-sft
LoRA model: /root/autodl-tmp/outputs-beta-dpo
Loading LoRA for causal language model
Loading checkpoint shards: 100%|██████████████████| 4/4 [00:23<00:00,  5.84s/it]
Merging with merge_and_unload...
Saving to Hugging Face format...
Done! model saved to /root/autodl-tmp/merged-ecom-adaptive-dpo
✅ DPO 阶段完成，最终模型已保存到 /root/autodl-tmp/merged-ecom-adative-dpo


In [52]:
!python merge_peft_adapter.py \
    --base_model {DPO_BASE_MODEL} \
    --lora_model {WORK_DIR}/outputs-orpo-qwen-v1 \
    --output_dir {WORK_DIR}/merged-ecom-orpo

Namespace(base_model='/root/autodl-tmp/merged-ecom-sft', tokenizer_path=None, lora_model='/root/autodl-tmp/outputs-orpo-qwen-v1', resize_emb=False, output_dir='/root/autodl-tmp/merged-ecom-orpo', hf_hub_model_id='', hf_hub_token=None)
Base model: /root/autodl-tmp/merged-ecom-sft
LoRA model: /root/autodl-tmp/outputs-orpo-qwen-v1
Loading LoRA for causal language model
Loading checkpoint shards: 100%|██████████████████| 4/4 [00:05<00:00,  1.45s/it]
Merging with merge_and_unload...
Saving to Hugging Face format...
Done! model saved to /root/autodl-tmp/merged-ecom-orpo


---

## Module 6️⃣ 综合评估

多维度对比评估 SFT 模型和 DPO 模型的效果差异

In [ ]:
# ============================================================
# 6.1 生成评估数据
# ============================================================
!python generate_eval_dataset.py \
    --output data/eval/ecommerce_eval.jsonl \
    --num 100 \
    --show_stats

In [ ]:
# ============================================================
# 6.2 运行评估
# ============================================================
!python evaluation_system.py \
    --models SFT DPO beta-DPO ORPO \
    --model_paths \
        {SFT_MERGED_DIR} \
        /root/autodl-tmp/merged-ecom-dpo \
        /root/autodl-tmp/merged-ecom-adaptive-dpo \
        {WORK_DIR}/merged-ecom-orpo \
    --eval_data ./data/eval/ecommerce_eval.jsonl \
    --output_dir eval_results/ \
    --max_new_tokens 512

2026-03-14 23:44:34,703 - INFO - 加载评估问题 60 条
2026-03-14 23:44:34,704 - INFO - 加载模型 [SFT]: /root/autodl-tmp/merged-ecom-sft
2026-03-14 23:44:37,260 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
Loading checkpoint shards: 100%|██████████████████| 4/4 [00:05<00:00,  1.44s/it]
/root/miniconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/root/miniconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset

In [7]:
!cd /root/autodl-tmp/

In [9]:
# SFT 模型
!python -m fastchat.llm_judge.gen_model_answer \
    --model-path /root/autodl-tmp/merged-ecom-sft \
    --model-id ecom-sft \
    --bench-name mt_bench

# DPO 模型
!python -m fastchat.llm_judge.gen_model_answer \
    --model-path /root/autodl-tmp/merged-ecom-dpo \
    --model-id ecom-dpo \
    --bench-name mt_bench

# ORPO 模型
!python -m fastchat.llm_judge.gen_model_answer \
    --model-path /root/autodl-tmp/merged-ecom-orpo \
    --model-id ecom-orpo \
    --bench-name mt_bench

Output to data/mt_bench/model_answer/ecom-sft.jsonl
  0%|                                                    | 0/80 [00:00<?, ?it/s]/root/miniconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/root/miniconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/root/miniconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do

In [10]:
import os
os.environ["OPENAI_API_KEY"] = "sk-113e4c3e4e50429ab9b80376b3cd07df"
os.environ["OPENAI_API_BASE"] = "https://api.deepseek.com/v1"


In [20]:
import urllib.request
try:
    resp = urllib.request.urlopen("https://api.deepseek.com", timeout=10)
    print("OK", resp.status)
except Exception as e:
    print("FAIL", e)

FAIL HTTP Error 401: Unauthorized


In [19]:
!python -m fastchat.llm_judge.gen_judgment \
    --model-list ecom-sft ecom-dpo ecom-orpo \
    --bench-name mt_bench \
    --judge-model deepseek-chat \
    --mode single \
    --parallel 5

Stats:
{
    "bench_name": "mt_bench",
    "mode": "single",
    "judge": "deepseek-chat",
    "baseline": null,
    "model_list": [
        "ecom-sft",
        "ecom-dpo",
        "ecom-orpo"
    ],
    "total_num_questions": 80,
    "total_num_matches": 480,
    "output_path": "data/mt_bench/model_judgment/deepseek-chat_single.jsonl"
}
Press Enter to confirm...^C
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/root/miniconda3/lib/python3.12/site-packages/fastchat/llm_judge/gen_judgment.py", line 304, in <module>
    input("Press Enter to confirm...")
KeyboardInterrupt


---

## Module 7️⃣ 最终效果测试

使用 DPO 后的最终模型进行全场景测试

In [49]:
# ============================================================
# 7.1 加载最终模型
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

final_model_path = /root/autodl-tmp/merged-ecom-dpo
tokenizer_final = AutoTokenizer.from_pretrained(final_model_path, trust_remote_code=True)
model_final     = AutoModelForCausalLM.from_pretrained(
    final_model_path, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
model_final.eval()

def final_chat(question: str, history: list = None,
               system: str = "你是专业的电商运营和销售顾问。") -> str:
    """支持多轮对话的推理函数"""
    messages = [{"role": "system", "content": system}]
    if history:
        for h_q, h_a in history:
            messages.append({"role": "user", "content": h_q})
            messages.append({"role": "assistant", "content": h_a})
    messages.append({"role": "user", "content": question})
    text = tokenizer_final.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer_final(text, return_tensors="pt").to(model_final.device)
    with torch.no_grad():
        output = model_final.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )
    new_ids = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer_final.decode(new_ids, skip_special_tokens=True)

SyntaxError: invalid syntax (700598464.py, line 7)

In [ ]:
# ============================================================
# 7.2 单轮全场景测试
# ============================================================
FINAL_TESTS = {
    "📝 文案生成": [
        "帮我写一个无线蓝牙耳机的爆款淘宝标题，要求高点击率",
        "为'便携空气炸锅'写一篇小红书种草文案，用真实用户视角",
    ],
    "💬 客服话术": [
        "买家说：'你家耳机质量怎么样？和苹果比哪个好？'，我该怎么回？",
        "买家执意要砍价30元否则不买，客服如何应对守住价格？",
        "买家问你家这个电热毯安全吗？会不会漏电？"
    ],
    "⭐ 差评处理": [
        "收到差评：'东西是假货，质量极差，骗人的店铺'，商家如何专业回复？",
    ],
    "📈 运营策略": [
        "新品上架零销量零评价，如何在7天内快速冷启动？",
        "商品点击率10%但转化率只有0.8%，该怎么优化？",
    ],
    "🎯 促销方案": [
        "为双11设计一个客单价从100元提升到200元的满减策略",
    ],
}

for category, questions in FINAL_TESTS.items():
    print(f"\n{'='*60}")
    print(f"  {category}")
    print(f"{'='*60}")
    for q in questions:
        print(f"\n❓ {q}")
        print("-" * 40)
        ans = final_chat(q)
        print(ans)
        print()

In [ ]:
# ============================================================
# 7.3 多轮对话测试
# ============================================================
print("=" * 60)
print("💬 多轮对话实战测试 — 售前咨询场景")
print("=" * 60)

history = []
dialogue_script = [
    "我想在淘宝上卖无线蓝牙耳机，但是竞争太激烈了，怎么差异化？",
    "你说的这些我都大概了解，但我的预算有限只有3万，应该先做哪些？",
    "直通车和超级推荐怎么选？我是新店没有数据基础",
    "好的，那你帮我制定一个第一个月的运营计划表吧",
]

for q in dialogue_script:
    print(f"\n🧑 买家：{q}")
    print("-" * 40)
    ans = final_chat(q, history=history)
    print(f"🤖 顾问：{ans}")
    history.append((q, ans))

print("\n" + "=" * 60)
print("💬 多轮对话实战测试 — 砍价博弈场景")
print("=" * 60)

history2 = []
bargain_script = [
    "这个蓝牙耳机299太贵了，能便宜点吗？",
    "别家同款才199，你们凭什么卖这么贵？",
    "那你给我打个八折吧，不然我就去别家了",
    "行吧，那送个耳机壳和延保可以吗？",
]

for q in bargain_script:
    print(f"\n🧑 买家：{q}")
    print("-" * 40)
    ans = final_chat(q, history=history2,
                     system="你是淘宝客服，负责销售无线蓝牙耳机（售价299元）。要维护价格体系同时留住客户。")
    print(f"🤖 客服：{ans}")
    history2.append((q, ans))

---

## 📌 附录

### 显存优化参考

| GPU 显存 | 推荐模型 | batch_size | gradient_accumulation | lora_rank |
|---------|---------|-----------|----------------------|----------|
| 8 GB    | Qwen2.5-0.5B | 1 | 8 | 8 |
| 16 GB   | Qwen2.5-3B   | 2 | 4 | 8 |
| 24 GB   | Qwen2.5-7B   | 2 | 4 | 16 |
| 40 GB   | Qwen2.5-14B  | 4 | 2 | 16 |
| 80 GB   | Qwen2.5-72B  | 4 | 1 | 32 |

### 关键超参说明

- **SFT learning_rate**: 建议 `1e-5 ~ 5e-5`，太大容易过拟合
- **DPO learning_rate**: 建议 `1e-7 ~ 5e-6`，DPO 对学习率非常敏感
- **DPO beta**: 控制偏好强度，`0.1~0.5`，越大越激进
- **lora_rank**: 数据量少用 8，数据量大用 16~64
- **gradient_checkpointing**: 显存不足时开启，会略微降低速度

### 文件清单

| 文件 | 用途 |
|------|------|
| `generate_ecommerce_dataset_v3.py` | 单轮 SFT + DPO 数据生成 |
| `multiturn_dialogue.py` | **[新增]** 多轮对话数据生成 |
| `data_quality_pipeline.py` | 数据质量清洗 |
| `sft_with_llamafactory.py` | **[新增]** LLaMA-Factory SFT 训练 |
| `dpo_training.py` | 标准 DPO 训练 |
| `advanced_alignment.py` | **[更新]** Adaptive-β DPO + ORPO |
| `merge_peft_adapter.py` | LoRA 权重合并 |
| `evaluation_system.py` | 综合评估 |
| `generate_eval_dataset.py` | 评估数据生成 |